In [ ]:
# Ensure the proteus package is importable from the notebooks/ subdirectory
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

# Lab 2: Build the Memory Engine

**Vector Search Foundations → Memory Architecture → MemoryManager & Toolbox**

In this lab you'll set up the full memory infrastructure that powers Proteus:
connect to the database, create vector-enabled tables, ingest research papers,
build the MemoryManager, and register tools into the semantic Toolbox.

By the end, you'll have a working memory engine you can query from Python.

## Task 1: Restore credentials and connect

In [ ]:
import os, logging

# Restore credentials from Lab 1
%store -r adb_dsn vector_user vector_password

# Set environment variables so proteus modules can read them
os.environ["PROTEUS_DB_DSN"] = adb_dsn
os.environ["PROTEUS_DB_PASSWORD"] = vector_password
os.environ["PROTEUS_DB_USER"] = vector_user

In [ ]:
from proteus.db import connect_to_oracle

vector_conn = connect_to_oracle()
print("Using user:", vector_conn.username)

## Task 2: Initialize embeddings and create the vector search demo table

In [ ]:
from proteus.vector_store import get_embedding_model, create_vector_store, safe_create_index
from proteus import config

embedding_model = get_embedding_model()

# Create a demo vector store for exploring search
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_oracledb.vectorstores import OracleVS

vector_store = OracleVS(
    client=vector_conn,
    embedding_function=embedding_model,
    table_name="VECTOR_SEARCH_DEMO",
    distance_strategy=DistanceStrategy.COSINE,
)

safe_create_index(vector_conn, vector_store, "oravs_hnsw")

## Task 3: Ingest research papers from arXiv

In [ ]:
from proteus.vector_store import ingest_arxiv_papers

# This may take 3-5 minutes for 300 papers
sampled_papers = ingest_arxiv_papers(vector_store, max_papers=300)
print(f"\nSample paper: {sampled_papers[0]['title'][:80]}...")

In [ ]:
# Inspect metadata fields available for filtering
sample_primary_subject = sampled_papers[0]["primary_subject"] if sampled_papers else ""
sample_arxiv_id = sampled_papers[0]["arxiv_id"] if sampled_papers else ""
print("Sample primary subject:", sample_primary_subject)
print("Sample arxiv_id:", sample_arxiv_id)

## Task 4: Query with natural language

Semantic search finds documents based on *meaning*. A query about "planetary exploration"
will match relevant papers even if those exact words don't appear in the title.

In [ ]:
# Basic similarity search
query = "Find research papers about planetary exploration mission planning"
results = vector_store.similarity_search(query, k=3)

for i, doc in enumerate(results, start=1):
    print(f"--- Result {i} ---")
    print("Title:", doc.metadata.get("title", "N/A"))
    print("Subject:", doc.metadata.get("primary_subject", "N/A"))
    print("Text:", doc.page_content[:200], "...")
    print()

In [ ]:
# Search with relevance scores (lower = more similar for cosine distance)
query = "methods for detecting gravitational waves"
results = vector_store.similarity_search_with_score(query, k=3)

for doc, score in results:
    print(f"Score: {score:.4f}")
    print(f"Title: {doc.metadata.get('title', 'N/A')}")
    print(f"Subject: {doc.metadata.get('primary_subject', 'N/A')}")
    print("------")

## Task 5: Filtered search with metadata

In [ ]:
# Filter by subject area
query = "Find papers related to mission planning and observational astronomy"
docs = vector_store.similarity_search(
    query, k=3,
    filter={"primary_subject": {"$eq": sample_primary_subject}},
)
for doc in docs:
    print("Title:", doc.metadata.get("title", "N/A"))
    print("Subject:", doc.metadata.get("primary_subject", "N/A"))
    print("------")

In [ ]:
# Filter by paper ID
docs = vector_store.similarity_search(
    query="Explain key themes in this research paper",
    k=5,
    filter={"id": {"$in": [sample_arxiv_id]}},
)
for doc in docs:
    print("Title:", doc.metadata.get("title", "N/A"))
    print("ArXiv ID:", doc.metadata.get("arxiv_id", "N/A"))
    print("------")

---

## Task 6: Create all memory tables and vector stores

Now we move from the demo table to the full memory architecture.
This creates 7 tables: conversational memory, tool log (both SQL),
plus 5 vector-enabled tables for knowledge base, workflow, toolbox, entity, and summary.

In [ ]:
from proteus.db import drop_all_tables, create_conversational_history_table, create_tool_log_table
from proteus.vector_store import create_all_vector_stores

# Drop any existing tables and start fresh
drop_all_tables(vector_conn)

# Create SQL tables
CONVERSATION_HISTORY_TABLE = create_conversational_history_table(vector_conn)
TOOL_LOG_TABLE_NAME = create_tool_log_table(vector_conn)

# Create all vector stores with HNSW indexes
stores = create_all_vector_stores(vector_conn, embedding_model)
print("\nStores created:", list(stores.keys()))

## Task 7: Seed the knowledge base with research papers

In [ ]:
from proteus.vector_store import seed_knowledge_base

# This may take 3-5 minutes
seed_knowledge_base(stores["knowledge_base_vs"], sampled_papers)

---

## Task 8: Initialize the MemoryManager

In [ ]:
from proteus.memory_manager import MemoryManager

memory_manager = MemoryManager(
    conn=vector_conn,
    conversation_table=CONVERSATION_HISTORY_TABLE,
    knowledge_base_vs=stores["knowledge_base_vs"],
    workflow_vs=stores["workflow_vs"],
    toolbox_vs=stores["toolbox_vs"],
    entity_vs=stores["entity_vs"],
    summary_vs=stores["summary_vs"],
    tool_log_table=TOOL_LOG_TABLE_NAME,
)
print("✅ MemoryManager initialized")

### Quick smoke test

In [ ]:
# Write a test conversation
test_thread = "SESSION-TEST-001"
memory_manager.write_conversational_memory(
    "Can you find papers on transformer architectures for time-series forecasting?",
    "user", test_thread
)
memory_manager.write_conversational_memory(
    "Let me search the knowledge base for relevant papers.",
    "assistant", test_thread
)

# Read it back
print(memory_manager.read_conversational_memory(test_thread))

In [ ]:
# Test knowledge base search
print(memory_manager.read_knowledge_base("transformer architectures for time-series"))

---

## Task 9: Initialize the Toolbox and register a test tool

In [ ]:
import getpass

def set_env_securely(var_name, prompt):
    value = getpass.getpass(prompt)
    os.environ[var_name] = value

set_env_securely("OPENAI_API_KEY", "OpenAI API Key: ")

In [ ]:
from openai import OpenAI
from proteus.toolbox import Toolbox

client = OpenAI()
toolbox = Toolbox(memory_manager=memory_manager, llm_client=client, embedding_model=embedding_model)
print("✅ Toolbox initialized")

In [ ]:
# Register a sample tool with LLM-augmented docstring
@toolbox.register_tool(augment=True)
def lookup_paper_details(arxiv_id: str) -> str:
    """Look up detailed metadata for a research paper by its arXiv ID."""
    mock_papers = {
        "1706.03762": "Title: Attention Is All You Need | Authors: Vaswani et al. | Subject: cs.CL | Year: 2017",
        "2005.14165": "Title: Language Models are Few-Shot Learners | Authors: Brown et al. | Subject: cs.CL | Year: 2020",
        "2303.08774": "Title: GPT-4 Technical Report | Authors: OpenAI | Subject: cs.CL | Year: 2023",
    }
    return mock_papers.get(arxiv_id, f"❓ Paper not found: {arxiv_id}")

print("✅ Tool registered")

In [ ]:
# Test semantic retrieval — does the toolbox find this tool?
import pprint
retrieved_tools = memory_manager.read_toolbox("look up details for a specific research paper")
pprint.pprint(retrieved_tools)

✅ **Lab 2 complete!** You now have:
- A vector search demo with 300 arXiv papers
- All 7 memory tables created and indexed
- A working MemoryManager with read/write for every memory type
- A semantic Toolbox with LLM-augmented tool registration

**Next up:** Lab 3 — Context Engineering & Web Search